# 🏰 Obrázok → 3D model (Hunyuan3D-2)

Z obrázka budovy (napr. z Forge) spraví 3D model (Hunyuan3D-2mini tvar + Hunyuan3D-2 textúra) **s farbami** ako `.glb` súbor pre Godot.

**Postup:**
1. Skontroluj, že vpravo dole je napísané **T4**. Ak nie: v hornom menu **Runtime → Zmeniť typ runtime → T4 GPU → Uložiť**.
2. Klikni **▶ Spustiť všetko** (tlačidlo hore pod menu, alebo **Ctrl+F9**). Ak vyskočí varovanie, že notebook nie je od Googlu, daj **Aj tak spustiť**.
3. Bunka 1 sa po inštalácii (cca 5–10 min) sama reštartuje. Potom daj **znova Spustiť všetko** (Ctrl+F9). Druhý beh preskočí inštaláciu a stiahne modely. Čakaj, kým sa v 3. bunke neobjaví tlačidlo **Vybrať súbory**.
4. Vyber obrázok (alebo viac naraz). Pozadie sa odstráni samo.
5. Po pár minútach sa `.glb` stiahne sám (prehliadač sa môže spýtať, či povoliť viac súborov → povoliť).
6. Ak niečo zlyhá: v menu **Runtime → Odpojiť a odstrániť runtime**, potom znova Ctrl+F9.
7. Ďalší obrázok: klikni na ▶ pri bunke **3** (netreba spúšťať všetko znova).

**Tipy:** Najlepšie funguje jedna budova v strede, celá v zábere, na jednoduchom pozadí, mierne zhora zboku.
Ak sa Colab odpojí alebo napíše, že došla GPU, skús neskôr (bezplatný limit je pár hodín denne).

In [ ]:
#@title 1. Inštalácia (raz za spustenie, cca 5–10 min)
import os, subprocess, sys
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("❌ Nie je zapnutá GPU! Runtime → Change runtime type → T4 GPU → Save, potom znova Run all.")
print("✅ GPU:", gpu)

def sh(cmd, povinne=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and povinne:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit("❌ Chyba pri: " + cmd)

ZNACKA = "/content/.instalacia_ok"
if os.path.exists(ZNACKA):
    print("✅ Inštalácia už je hotová, pokračujem")
else:
    os.chdir("/content")
    if not os.path.exists("/content/Hunyuan3D-2"):
        sh("git clone --depth 1 https://github.com/Tencent/Hunyuan3D-2")
    os.chdir("/content/Hunyuan3D-2")
    print("⏳ Inštalujem knižnice...")
    # Už nainštalované knižnice (numpy, Pillow, torch...) nesmú zmeniť verziu počas behu,
    # inak padá import. Meniť sa smú len tieto:
    menit = {"diffusers", "transformers", "huggingface-hub", "huggingface_hub", "tokenizers", "safetensors", "accelerate", "hf-xet", "hf_xet"}
    zoznam = subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"], capture_output=True, text=True).stdout
    riadky = [r for r in zoznam.splitlines() if "==" in r and r.split("==")[0].lower() not in menit]
    open("/content/obmedzenia.txt", "w").write("\n".join(riadky) + "\n")
    sh("pip install -q -c /content/obmedzenia.txt diffusers==0.32.2 transformers==4.49.0 'huggingface_hub<0.31' "
       "accelerate einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime ninja pybind11")
    sh("pip install -q -e . --no-deps")
    print("⏳ Kompilujem renderer textúr (najdlhší krok)...")
    os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
    sh("pip install -q --no-build-isolation ./hy3dgen/texgen/custom_rasterizer")
    sh("cd hy3dgen/texgen/differentiable_renderer && python setup.py build_ext --inplace", povinne=False)
    print("⏳ Kontrolujem knižnice...")
    test = "from PIL import Image; import numpy, diffusers; from transformers import CLIPImageProcessor; import custom_rasterizer"
    if subprocess.run([sys.executable, "-c", test], capture_output=True).returncode != 0:
        sh("pip install -q --force-reinstall --no-deps " + " ".join(r for r in riadky if r.lower().startswith(("pillow==", "numpy=="))))
    open(ZNACKA, "w").write("ok")
    print("✅ Inštalácia hotová. Prostredie sa teraz samo reštartuje.")
    print("👉 Počkaj pár sekúnd a daj ZNOVA ▶ Spustiť všetko (Run all).")
    import time; time.sleep(2)
    os.kill(os.getpid(), 9)

In [ ]:
#@title 2. Načítanie modelov (stiahne cca 8 GB, pár minút)
import os, sys, gc
os.environ["USE_TF"] = "0"              # nenačítavať TensorFlow (zbytočne berie pamäť)
os.environ["TRANSFORMERS_NO_TF"] = "1"
import torch
os.chdir("/content/Hunyuan3D-2")
sys.path.insert(0, "/content/Hunyuan3D-2")
from hy3dgen.rembg import BackgroundRemover
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline, FaceReducer, FloaterRemover, DegenerateFaceRemover
from hy3dgen.texgen import Hunyuan3DPaintPipeline

def pamat():
    import psutil
    print(f"   RAM {psutil.virtual_memory().used/1e9:.1f} GB, grafika {torch.cuda.memory_allocated()/1e9:.1f} GB")

rembg = BackgroundRemover()
print("⏳ Model tvaru...")
shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained("tencent/Hunyuan3D-2mini", subfolder="hunyuan3d-dit-v2-mini", variant="fp16")
gc.collect(); torch.cuda.empty_cache(); pamat()
print("⏳ Model textúr...")
paint_pipe = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2")
gc.collect(); torch.cuda.empty_cache(); pamat()
print("✅ Modely pripravené")

In [ ]:
#@title 3. Nahraj obrázok → stiahni GLB (spúšťaj znova pre ďalšie obrázky)
max_trojuholnikov = 30000  #@param {type:"integer"}
s_texturou = True  #@param {type:"boolean"}
kvalita = "normálna"  #@param ["rýchla", "normálna", "vysoká"]

import io, os, time, torch, gc
from PIL import Image
from google.colab import files

kroky, rozlisenie = {"rýchla": (25, 256), "normálna": (40, 320), "vysoká": (50, 384)}[kvalita]
os.makedirs("/content/vystup", exist_ok=True)
nahrane = files.upload()
for meno, data in nahrane.items():
    t0 = time.time()
    nazov = os.path.splitext(meno)[0]
    print(f"\n🏗️  {meno}")
    img = Image.open(io.BytesIO(data))
    if img.mode == "RGBA" and img.getextrema()[3][0] < 255:
        img = img.convert("RGBA")          # už má priehľadné pozadie
    else:
        img = rembg(img.convert("RGB"))    # odstráni pozadie
    try:
        mesh = shape_pipe(image=img, num_inference_steps=kroky, octree_resolution=rozlisenie,
                          num_chunks=8000, generator=torch.manual_seed(1234))[0]
        mesh = FloaterRemover()(mesh)
        mesh = DegenerateFaceRemover()(mesh)
        mesh = FaceReducer()(mesh, max_facenum=max_trojuholnikov)
        print(f"   tvar hotový ({time.time()-t0:.0f} s)")
        gc.collect(); torch.cuda.empty_cache()
        cesta = f"/content/vystup/{nazov}.glb"
        if s_texturou:
            try:
                mesh = paint_pipe(mesh, image=img)
                print(f"   textúra hotová ({time.time()-t0:.0f} s)")
            except Exception as e:
                print("   ⚠️ Textúra zlyhala, ukladám bez farieb:", e)
                cesta = f"/content/vystup/{nazov}_bez_farieb.glb"
        mesh.export(cesta)
        files.download(cesta)
        print(f"✅ {os.path.basename(cesta)} ({time.time()-t0:.0f} s)")
    except torch.cuda.OutOfMemoryError:
        print("❌ Došla pamäť grafiky. Skús kvalitu 'rýchla' alebo Runtime → Restart session a Run all.")
    finally:
        gc.collect(); torch.cuda.empty_cache()